# Workable reviews and published embeddings

Load existing embeddings, select review sources, and keep each vector beside its original text. This notebook reads S3 releases through VOC; it does not encode reviews, run a pipeline, or publish results.

Copy this template into your private `notebooks/nb/` workspace before saving executed outputs. Use the existing VOC Jupyter environment and configured AWS access. No model downloads or OpenRouter credentials are needed.

## 1. Choose releases and sources

Both versions default to `"latest"`. VOC first resolves the dataset, then finds the latest promoted embeddings for **that dataset and profile**. If no matching embeddings have been published, the reader raises an error; it does not substitute another dataset or generate vectors.

Here, **source means platform**: `app_store`, `google_play`, `x`, or `youtube`. Choose one or more, or set `PLATFORMS = None` for all. `preferred_provider` remains available separately (for example, Appbot can supply App Store reviews).

This template requires full embedding coverage. Filtering happens after loading the verified release, so it does not reduce the initial download. No relevance labels are excluded automatically.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from voc import collector as co
from voc.embeddings import get_embeddings

DATASET_VERSION = "latest"
EMBEDDING_VERSION = "latest"
PROFILE = "minilm-verbatim-v1"

PLATFORMS = ["youtube"]  # None = all; example: ["app_store", "google_play"]

AVAILABLE_PLATFORMS = {"app_store", "google_play", "x", "youtube"}
if PLATFORMS is not None:
    if not isinstance(PLATFORMS, list) or not PLATFORMS:
        raise ValueError("PLATFORMS must be None or a nonempty list of platform names")
    if any(not isinstance(value, str) for value in PLATFORMS):
        raise ValueError("Every platform name must be a string")
    unknown = set(PLATFORMS) - AVAILABLE_PLATFORMS
    if unknown:
        raise ValueError(f"Unknown platforms: {sorted(unknown)}")


## 2. Load verified inputs

Keep the snapshot unmodified when passing it to the embedding reader. The reader validates the dataset identity, text, row alignment, artifact checksums and vector properties.

The resolved IDs below identify what this execution actually loaded. Copy them into the configuration when you need to reproduce this input instead of following future publications.

In [ ]:
snapshot = co.get_dataset(
    stage="prepared", source="s3", version=DATASET_VERSION,
)
bundle = get_embeddings(
    dataset=snapshot,
    profile=PROFILE,
    version=EMBEDDING_VERSION,
    source="s3",
    allow_subset=False,
)

assert len(bundle.reviews) == bundle.vectors.shape[0]
assert len(bundle.reviews) == len(snapshot.data)
assert bundle.info["selection"]["kind"] == "full"

resolved = {
    "dataset_release_id": bundle.info["dataset"]["release_id"],
    "embedding_release_id": bundle.info["embedding_release_id"],
    "profile": bundle.info["profile"]["name"],
    "profile_id": bundle.info["profile"]["profile_id"],
    "loaded_reviews": len(bundle.reviews),
    "dimensions": bundle.vectors.shape[1],
}
display(pd.Series(resolved, name="resolved input").to_frame())


## 3. Attach the original text and filter by source

The embedding bundle contains the exact `embedding_text` used by its producer. Recover `text` (the original review body), title, platform and other context from the workable dataset by `record_id`. Keep both text columns: the original body and encoder input can differ.

The join preserves vector order and requires one dataset row per review. Platform counts show which sources are available before filtering.

In [ ]:
context_columns = [
    "record_id", "platform", "preferred_provider", "text", "title",
    "relevance", "record_date",
]
all_reviews = bundle.reviews.merge(
    snapshot.data[context_columns],
    on="record_id", how="left", sort=False, validate="one_to_one", indicator=True,
)
assert all_reviews["_merge"].eq("both").all()
all_reviews = all_reviews.drop(columns="_merge").reset_index(drop=True)
assert all_reviews["record_id"].tolist() == bundle.reviews["record_id"].tolist()
assert all_reviews["vector_row"].tolist() == list(range(len(all_reviews)))

display(all_reviews.groupby("platform").size().rename("available_reviews").to_frame())


In [ ]:
mask = (
    np.ones(len(all_reviews), dtype=bool)
    if PLATFORMS is None
    else all_reviews["platform"].isin(PLATFORMS).to_numpy()
)
positions = np.flatnonzero(mask)
df = all_reviews.iloc[positions].copy().reset_index(drop=True)
embeddings = (
    bundle.vectors if len(positions) == len(all_reviews)
    else bundle.vectors[positions]
)

# Each dataframe cell holds a NumPy vector; row i matches embeddings[i].
df["embedding"] = list(embeddings)
assert len(df) == embeddings.shape[0]
assert df["record_id"].tolist() == bundle.reviews.iloc[positions]["record_id"].tolist()
assert df["vector_row"].tolist() == positions.tolist()

print(f"Selected {len(df):,} of {len(all_reviews):,} reviews")
print("Embedding matrix:", embeddings.shape)
if df.empty:
    print("No reviews match these sources in this release. Choose another filter.")


## 4. Inspect the dataframe

`df` includes `record_id`, `platform`, `text`, `embedding_text`, `relevance` and an `embedding` column containing NumPy vectors. It also retains context and embedding provenance columns.

For readability, the preview shows only the first five vector values. The actual `df["embedding"]` values retain every dimension. Inspect source and relevance counts; a negative sentiment and an irrelevant review are different concepts, and this notebook computes neither.

In [ ]:
display_columns = [
    "record_id", "platform", "preferred_provider", "text", "embedding_text", "relevance",
]
preview = df[display_columns].head(5).copy()
preview["embedding_preview"] = [vector[:5].round(4).tolist() for vector in embeddings[:5]]
with pd.option_context("display.max_colwidth", 100):
    display(preview)

display(df.groupby("platform").size().rename("selected_reviews").to_frame())
display(df["relevance"].value_counts(dropna=False).rename("reviews").to_frame())


## 5. Continue your analysis

You now have two aligned outputs:

- `df`: filtered reviews, original body text, encoder input, context, and vectors.
- `embeddings`: the corresponding NumPy matrix, with one row per dataframe row.

Use `texts` below with `embeddings` when an algorithm expects the encoder's input text and vector matrix separately. Use `df["text"]` to inspect the original body.

The dataframe index describes the filtered order. `vector_row` preserves the row number in the original published matrix; after filtering, it may have gaps. Always apply subsequent sorting/filtering to both outputs together, or rebuild the matrix from the dataframe's embedding column. Treat the published vectors as read-only.

After changing configuration, run the notebook from the beginning. Nothing is exported or published by these cells.

In [ ]:
texts = df["embedding_text"].tolist()
assert len(texts) == len(df) == embeddings.shape[0]
if not df.empty:
    np.testing.assert_array_equal(df.iloc[0]["embedding"], embeddings[0])
print("Ready for analysis:", len(texts), "reviews and", embeddings.shape[1], "dimensions")
